#XAI testing

# Import drive and config

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!rm -r /content/LibEER
!git clone https://github.com/jitalo333/LibEER

rm: cannot remove '/content/LibEER': No such file or directory
Cloning into 'LibEER'...
remote: Enumerating objects: 716, done.
remote: Counting objects: 100% (255/255), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 716 (delta 186), reused 165 (delta 114), pack-reused 461 (from 1)
Receiving objects: 100% (716/716), 1.05 MiB | 22.29 MiB/s, done.
Resolving deltas: 100% (403/403), done.


# Install

In [3]:
import os

In [4]:
path_repo='/content/LibEER'
os.chdir(path_repo)
print("Directorio de trabajo actual:", os.getcwd())
!pip install -r requirements.txt

Directorio de trabajo actual: /content/LibEER
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.5/268.5 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 448.8/448.8 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.4/178.4 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 146.7 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERR

In [5]:
path_repo='/content/LibEER/LibEER'
os.chdir(path_repo)
print("Directorio de trabajo actual:", os.getcwd())

Directorio de trabajo actual: /content/LibEER/LibEER


# Subject dependent

## Load data

In [6]:
class Setting_Custom:
    def __init__(self, experiment_mode_num=1, split_type_num=1):
        """
        Parámetros:
        experiment_mode_num:
            1 -> subject-independent
            2 -> subject-dependent
            3 -> cross-session

        split_type_num:
            1 -> leave-one-out
            2 -> kfold
            3 -> front-back
            4 -> train-val-test
        """

        # --- Mapeo numérico a texto ---
        experiment_modes = {
            1: "subject-independent",
            2: "subject-dependent",
            3: "cross-session"
        }
        split_types = {
            1: "leave-one-out",
            2: "kfold",
            3: "front-back",
            4: "train-val-test"
        }

        # --- Asignaciones ---
        self.experiment_mode = experiment_modes.get(experiment_mode_num)
        self.split_type = split_types.get(split_type_num)

        # --- Validaciones ---
        if self.experiment_mode is None:
            raise ValueError("experiment_mode_num debe ser 1 (subject-independent), 2 (subject-dependent) o 3 (cross-session).")
        if self.split_type is None:
            raise ValueError("split_type_num debe ser 1 (leave-one-out), 2 (kfold), 3 (front-back) o 4 (train-val-test).")

        # Restricción: subject-dependent no puede ir con leave-one-out
        if self.experiment_mode == "subject-dependent" and self.split_type == "leave-one-out":
            raise ValueError("❌ 'subject-dependent' no puede usarse con 'leave-one-out'.")

        # --- Parámetros de merge ---
        self.cross_trail = "true" ####### Dejarlo en true para separar tomando en cuenta los trials
        self.sessions = [1, 2, 3]
        self.pr = None

        # --- Parámetros de split ---
        self.fold_num = 5
        self.fold_shuffle = True
        self.seed = 2024
        self.front = 800
        self.test_size = 0.2
        self.val_size = 0.2
        self.sr = None

In [ ]:
import importlib
import data_utils.load_data as ld
importlib.reload(ld)

<module 'data_utils.load_data' from '/content/LibEER/LibEER/data_utils/load_data.py'>

In [7]:
from data_utils.load_data import read_escolares_preprocessed
from data_utils.load_data import get_uniform_data
from data_utils.preprocess import baseline_removal, bandpass_filter, feature_extraction, segment_data
import numpy as np


data_dir = '/content/drive/MyDrive/Colab Notebooks/SEED_dataset_completo/SEED_EEG' ################### CAMBIAR ESTO
unified_data, baseline, unified_label, sample_rate, channels = get_uniform_data(dataset="seed_de_lds", dataset_path=data_dir, test_mode = True)

# Dimension visualization
subject = 2
print(len(unified_data), len(unified_data[0]), len(unified_data[0][subject]),np.array(unified_data[0][subject][0]).shape)
print(len(unified_label), len(unified_label[0]), np.array(unified_label[0][subject]).shape)

3 15 15 (235, 62, 5)
3 15 (15,)


In [8]:
import os
import pickle
import numpy as np
#import functions
from data_utils.preprocess import label_process, reorder_channels_SEED
#import lists and dicts
from data_utils.preprocess import channels_DEAP, channels_SEED, transform_SEED

from data_utils.split import index_to_data, merge_to_part, get_split_index
from utils.utils import setup_seed


# Reorder channels
data = reorder_channels_SEED(unified_data, channels_SEED, channels_DEAP, transform_SEED)

#Gen labels
data, label, num_classes = label_process(data=data, label=unified_label, bounds=[3, 7], onehot=False, label_used=["valence"], binary = False)

# experiment_mode_num: 1=subject-independent, 2=subject-dependent, 3=cross-session
# split_type_num: 1=leave-one-out, 2=kfold, 3=front-back, 4=train-val-test
setting = Setting_Custom(experiment_mode_num=2, split_type_num=4)

# Fix random seed
setup_seed(2024)
# prepare data
m_data, m_label = merge_to_part(data, label, setting)
subject = 3

# Get subjects to delete
delete_subjects = [None]

# Print dimensions
print(len(m_data), len(m_data[0]), np.array(m_data[0][subject]).shape)
print(len(m_label), len(m_label[0]), np.array(m_label[0][subject]).shape)

[0, 3, 7, 5, 15, 17, 25, 23, 33, 35, 43, 41, 52, 58, 59, 45, 2, 4, 9, 11, 13, 21, 19, 27, 29, 31, 39, 37, 47, 49, 54, 60]
45 15 (238, 32, 5)
45 15 (238,)


In [ ]:
for idx, (data_i, label_i) in enumerate(zip(m_data, m_label)):
    # according to the data format and label,  the test size is 0.2 and the validation size is 0.2
    spi = get_split_index(data_i, label_i,  setting=setting)
    for jdx, (train_indexes, test_indexes, val_indexes) in enumerate(zip(spi['train'],spi['test'], spi['val'])):
        # organize the data according to the resulting index
        (train_data, train_label, val_data, val_label,  test_data, test_label) = index_to_data(data_i, label_i,  train_indexes, test_indexes, val_indexes)
        #print(train_data.shape, val_data.shape, test_data.shape)


## Train

In [ ]:
from models.DGCNN import DGCNN
from Trainer.Custom_training.pytorch_pipeline import Pytorch_Pipeline, get_loaders, get_metrics
from utils.save_results import save_models_and_metrics
from sklearn.metrics import f1_score
import numpy as np

savepath = '/content/drive/MyDrive/SEED /SEED'   ################### CAMBIAR ESTO

os.makedirs(savepath, exist_ok=True)

# 2. COMBINAR los parámetros fijos y variables
params = {"lr": 0.0015,
          "batch_size": 32,
          "dropout": 0.5,
          "scaler": None,
          'input_dim': 160,
          'in_channels': 5,
          'num_classes': 3,
          'num_electrodes': 32,
          'k': 2,
          'relu_is': 1,
          'layers': [64],
          }

scaler = 'None'
#------------- Hyperparameter search -------------------------------
# Search hyperparameters for the model on all the selected subjects
F1 = []
all_metrics = []
for idx, (data_i, label_i) in enumerate(zip(m_data, m_label)):
    # according to the data format and label,  the test size is 0.2 and the validation size is 0.2
    spi = get_split_index(data_i, label_i,  setting=setting)
    for jdx, (train_indexes, test_indexes, val_indexes) in enumerate(zip(spi['train'],spi['test'], spi['val'])):
        # organize the data according to the resulting index
        (X_train, y_train, X_val, y_val,  X_test, y_test) = index_to_data(data_i, label_i,  train_indexes, test_indexes, val_indexes)

    subject = f"subject{idx}"
    pipeline_mlp =  Pytorch_Pipeline(model_class=DGCNN, sample_weights_loss = True)
    #Set params
    pipeline_mlp.set_params(**params)
    #Set criterion
    pipeline_mlp.set_criterion(y_train)

    # ----------- Escalado de datos -----------
    X_train, X_val = pipeline_mlp.set_scaler_transform(scaler, X_train, X_val, dtype="MultiDim_TimeSeries")
    _, X_test = pipeline_mlp.set_scaler_transform(scaler, X_train, X_test, dtype="MultiDim_TimeSeries")
    # ------------- Loaders --------------------
    train_loader, val_loader = get_loaders(X_train, X_val, y_train, y_val, pipeline_mlp.batch_size)
    _, test_loader = get_loaders(X_train, X_test, y_train, y_test, pipeline_mlp.batch_size)
    # ---------- Early stopping (por loss) ----------
    patience = 10
    min_delta = 1e-4
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_state = None

    for epoch in range(pipeline_mlp.max_epochs):
    #for epoch in range(2):
          pipeline_mlp.partial_fit(train_loader)
          avg_val_loss, f1, y_val, y_pred = pipeline_mlp.predict_and_evaluate(val_loader)
          # ---------- Optuna pruning with F1 ----------
          #Prune only on the first fold
          """
          if idx_subject == 0:
            trial.report(f1, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
          """
          # ---------- Early stopping (por loss) ----------
          if avg_val_loss + min_delta < best_val_loss:
              best_val_loss = avg_val_loss
              best_model_state = pipeline_mlp.model.state_dict()
              epochs_no_improve = 0
          else:
              epochs_no_improve += 1
              if epochs_no_improve >= patience:
                  break

    print("subject:", subject, "epoch:", epoch)
    #---------- Load best model ------------------------
    pipeline_mlp.model.load_state_dict(best_model_state)
    #----------Eval best model on the val set ---------
    _, f1, y_test, y_pred = pipeline_mlp.predict_and_evaluate(val_loader)
    metrics = get_metrics(y_test, y_pred)
    results_val = {
        'DGCNN': metrics
    }

    #----------Eval best model on the test set ---------
    _, f1, y_test, y_pred = pipeline_mlp.predict_and_evaluate(test_loader)
    #---------------- Save final result ----------------
    F1.append(f1)
    #-------------Visualization metrics-----------------
    metrics = get_metrics(y_test, y_pred)
    all_metrics.append(metrics)

    results_test = {
        'DGCNN': metrics
    }

    models_dicc = {
        'DGCNN': pipeline_mlp.model
    }

    filename = os.path.join(savepath, subject)
    os.makedirs(filename, exist_ok=True)
    save_models_and_metrics(filename, results_val = results_val, models_dicc = models_dicc, df_test = None, y_test=y_test,
                            results_test=results_test, preds_test = y_pred, save_preds = None)

#------------ Compute avg among 5 folds ----------------
mean_F1 = np.mean(F1)
print(mean_F1)

## Testing XAI repository

In [25]:
#load model and data:
import joblib
import os

#results_EEG_1s : DGCNN
#results_1s_CDCN : CDCN
savepath = '/content/drive/MyDrive/Colab Notebooks/results_EEG_1s'
subject_to_load = 'subject0' #subject0 se refiere a resultados de 1 persona, o resultados de 1 sesión de una persona? o resultados de un trial de una persona?

model_filename = os.path.join(savepath, subject_to_load, 'models', 'DGCNN.joblib')

try:
    # Assuming DGCNN.joblib contains the model object directly
    loaded_model = joblib.load(model_filename)
    print(f"Successfully loaded model for {subject_to_load}:")
    print(f"- Type of loaded model: {type(loaded_model)}")
except FileNotFoundError:
    print(f"Error: Model file not found at {model_filename}. Please ensure the training step was completed successfully and the subject exists.")
except Exception as e:
    print(f"An error occurred while loading the model: {e}")

Successfully loaded model for subject0:
- Type of loaded model: <class 'models.DGCNN.DGCNN'>


In [26]:
print(loaded_model)

DGCNN(
  (graphConvs): ModuleList(
    (0): GraphConv()
  )
  (fc): Linear(in_features=2048, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=3, bias=True)
  (relu): ReLU(inplace=True)
  (b_relus): ModuleList(
    (0): B1ReLU(
      (relu): ReLU()
    )
  )
  (dropout): Dropout(p=0.5, inplace=False)
)


In [10]:
from data_utils.preprocess import channels_DEAP

bands = ['Delta', 'Theta', 'Alpha', 'Beta', 'Gamma']

#obtener par canal_banda
feature_names_auto = []
for ch in channels_DEAP:
    for b in bands:
        feature_names_auto.append(f"{ch}_{b}")

print(f"Total de características: {len(feature_names_auto)}")
print(f"Ejemplo de nombres: {feature_names_auto[:5]}")

Total de características: 160
Ejemplo de nombres: ['fp1_Delta', 'fp1_Theta', 'fp1_Alpha', 'fp1_Beta', 'fp1_Gamma']


In [27]:
idx_subject = 0 # "subject 0", tiene que ser el mismo con el que se cargó el modelo

spi = get_split_index(m_data[idx_subject], m_label[idx_subject], setting=setting)

# primer fold (jdx=0)
train_idx, test_idx, val_idx = spi['train'][0], spi['test'][0], spi['val'][0]

X_train_sub0, y_train_sub0, X_val_sub0, y_val_sub0, X_test_sub0, y_test_sub0 = index_to_data(
    m_data[idx_subject], m_label[idx_subject], train_idx, test_idx, val_idx
)

print(f"X_test shape: {X_test_sub0.shape}")
print(f"y_test shape: {y_test_sub0.shape}")

X_test shape: (586, 32, 5)
y_test shape: (586,)


## Clone XAI Repository

In [36]:
path_repo='/content/'
os.chdir(path_repo)
print("Directorio de trabajo actual:", os.getcwd())
!ls

Directorio de trabajo actual: /content
drive  LibEER  sample_data  XAI-EEG-emotion-classification


In [37]:
#gradient specific branch
!rm -r /content/XAI-EEG-emotion-classification
!git clone -b antonia --single-branch https://github.com/jitalo333/XAI-EEG-emotion-classification.git

Cloning into 'XAI-EEG-emotion-classification'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 103 (delta 44), reused 78 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 20.39 KiB | 10.19 MiB/s, done.
Resolving deltas: 100% (44/44), done.


In [38]:
path_repo='/content/XAI-EEG-emotion-classification'
os.chdir(path_repo)
print("working path:", os.getcwd())
!pip install -r requirements.txt

working path: /content/XAI-EEG-emotion-classification
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.8/201.8 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 139.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 12.2 MB/s eta 0

In [31]:
import sys
import os
import torch
import importlib

path_xai = '/content/XAI-EEG-emotion-classification'
path_libeer = '/content/LibEER'

if path_xai not in sys.path: sys.path.append(path_xai)
if path_libeer not in sys.path: sys.path.append(path_libeer)

os.chdir(path_xai)
print(f"work path: {os.getcwd()}")

work path: /content/XAI-EEG-emotion-classification


## XAI Testing Methods

These methods require loaded data and a loaded model to work.
Local methods only explain one sample, while global explain all X_test.

'target_class' parameter in global methods are optional. If given a class value, the method will explain all the correct predictions of that particular class.

###Saliency maps

In [32]:
#saliency maps or vainilla gradient method, both local and global
try:
    from data_utils.preprocess import channels_DEAP
    print("channels_DEAP successfully loaded from LibEER")
except ImportError as e:
    print(f"Error loading LibEER: {e}. Ensure '/content/LibEER' exists.")

import torch
import os
import importlib

# Correctly reloading the Saliency Map module
import method1.model
importlib.reload(method1.model)
from method1.model import saliency_map

# 1. Hardware and Model Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model.to(device)
loaded_model.eval()

# 2. Base Parameters
verbose = True
channels = channels_DEAP
subject_id = 0
model_type = loaded_model.__class__.__name__

save_path = f"/content/drive/MyDrive/Colab Notebooks/plots_{model_type}_results"
os.makedirs(save_path, exist_ok=True)

print(f"Model type detected: {model_type}")
print(f"Results folder: {save_path}")

# ==================== LOCAL MODE (Single Sample) ====================
sample_idx = 0
input_data = X_test_sub0[sample_idx]

args_local = {
    'channels': channels,
    'subject_id': subject_id,
    'sample_idx': sample_idx,
    'input_data': input_data,
    'model_type': model_type,
    'is_global': False
}

print(f"\n1. Running LOCAL Saliency Map (Sample {sample_idx})...")
try:
    saliency_map(loaded_model, save_path, verbose, args_local)
except Exception as e:
    print(f"Error in Local Saliency Map: {e}")
    import traceback
    traceback.print_exc()

# ==================== GLOBAL MODE (Average over Test Set) ====================
args_global = {
    'channels': channels,
    'subject_id': subject_id,
    'X_test': X_test_sub0,  # Full test set
    'y_test': y_test_sub0,  # Labels for validation
    'model_type': model_type,
    'is_global': True,
    'target_class': 2       # Use the active class (0 or 1)
}

print("\n2. Running GLOBAL Saliency Map (Processing Test Set)...")
try:
    # This will use the tqdm progress bar from your model.py
    saliency_map(loaded_model, save_path, verbose, args_global)
    print(f"\nExecution finished. Results saved in: {save_path}")
except Exception as e:
    print(f"Error in Global Saliency Map: {e}")
    import traceback
    traceback.print_exc()

channels_DEAP successfully loaded from LibEER
Model type detected: DGCNN
Results folder: /content/drive/MyDrive/Colab Notebooks/plots_DGCNN_results

1. Running LOCAL Saliency Map (Sample 0)...
Before plot: 0.0 1.0

--- SALIENCY STATISTICS (DGCNN - Local) ---
Global Mean: 0.2207

--- TOP 5 CHANNELS ---
  af3: 0.6168
  fc1: 0.4276
  cp5: 0.4273
  p7: 0.3982
  f4: 0.3759

--- BAND IMPORTANCE ---
  Delta: 0.2065
  Theta: 0.1678
  Alpha: 0.1866
  Beta: 0.1893
  Gamma: 0.3535

2. Running GLOBAL Saliency Map (Processing Test Set)...


100%|██████████| 586/586 [00:00<00:00, 838.97it/s]



Averaged over 173 samples.
Averaged over 173 samples.
Before plot: 0.0732449 3.6620786
After plot norm: 0.0 1.0

--- SALIENCY STATISTICS (DGCNN - Global) ---
Global Mean: 0.7414

--- TOP 5 CHANNELS ---
  af3: 1.7860
  fc6: 1.5419
  fc1: 1.3850
  p7: 1.2329
  f4: 1.0314

--- BAND IMPORTANCE ---
  Delta: 0.6861
  Theta: 0.5790
  Alpha: 0.5269
  Beta: 0.6834
  Gamma: 1.2319

Execution finished. Results saved in: /content/drive/MyDrive/Colab Notebooks/plots_DGCNN_results


###InputxGradient

In [33]:
#local and global methods

try:
    from data_utils.preprocess import channels_DEAP
    print("channels_DEAP successfully loaded from LibEER")
except ImportError as e:
    print(f"Error loading LibEER: {e}. Ensure '/content/LibEER' exists.")

import torch
import os
import importlib

# Correctly reloading the Input x Gradient module
import method2.model
importlib.reload(method2.model)
from method2.model import input_x_gradient

# 1. Hardware and Model Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model.to(device)
loaded_model.eval()

# 2. Base Parameters
verbose = True
channels = channels_DEAP
subject_id = 0
model_type = loaded_model.__class__.__name__

save_path = f"/content/drive/MyDrive/Colab Notebooks/plots_{model_type}_results"
os.makedirs(save_path, exist_ok=True)

print(f"Model type detected: {model_type}")
print(f"Results folder: {save_path}")

# ==================== LOCAL MODE (Single Sample) ====================
sample_idx = 0
input_data = X_test_sub0[sample_idx]

args_local = {
    'channels': channels,
    'subject_id': subject_id,
    'sample_idx': sample_idx,
    'input_data': input_data,
    'model_type': model_type,
    'is_global': False
}

print(f"\n1. Running LOCAL Input x Gradient (Sample {sample_idx})...")
try:
    input_x_gradient(loaded_model, save_path, verbose, args_local)
except Exception as e:
    print(f"Error in Local Input x Gradient: {e}")
    import traceback
    traceback.print_exc()

# ==================== GLOBAL MODE (Average over Test Set) ====================
args_global = {
    'channels': channels,
    'subject_id': subject_id,
    'X_test': X_test_sub0,  # Full test set
    'y_test': y_test_sub0,  # Labels for validation
    'model_type': model_type,
    'is_global': True,
    'target_class': 2       # Matches the active class (0 or 1)
}

print("\n2. Running GLOBAL Input x Gradient (Processing Test Set)...")
try:
    # This will use the tqdm bar from your model.py
    input_x_gradient(loaded_model, save_path, verbose, args_global)
    print(f"\nExecution finished. Results saved in: {save_path}")
except Exception as e:
    print(f"Error in Global Input x Gradient: {e}")
    import traceback
    traceback.print_exc()

channels_DEAP successfully loaded from LibEER
Model type detected: DGCNN
Results folder: /content/drive/MyDrive/Colab Notebooks/plots_DGCNN_results

1. Running LOCAL Input x Gradient (Sample 0)...
Before plot: 0.0 1.0

--- Input_x_Gradient STATISTICS (DGCNN - Local) ---
Target/Pred Class: 2
Global Mean: 0.2343

--- TOP 5 CHANNELS ---
  af3: 0.6941
  cp5: 0.4351
  p7: 0.4306
  fc1: 0.4264
  fp1: 0.3968

--- BAND IMPORTANCE ---
  Delta: 0.2813
  Theta: 0.1937
  Alpha: 0.1966
  Beta: 0.1802
  Gamma: 0.3198

2. Running GLOBAL Input x Gradient (Processing Test Set)...


100%|██████████| 586/586 [00:00<00:00, 769.55it/s]



Averaged over 173 samples.
Before plot: 1.2893354 63.508106
After plot norm: 0.0 1.0

--- Input_x_Gradient STATISTICS (DGCNN - Global) ---
Target/Pred Class: 2
Global Mean: 14.5600

--- TOP 5 CHANNELS ---
  af3: 36.9118
  fc6: 28.9965
  fc1: 25.8455
  p7: 24.0662
  fp1: 18.9756

--- BAND IMPORTANCE ---
  Delta: 17.2809
  Theta: 12.2007
  Alpha: 10.2543
  Beta: 12.1571
  Gamma: 20.9072

Execution finished. Results saved in: /content/drive/MyDrive/Colab Notebooks/plots_DGCNN_results


###Integrated Gradients

In [34]:
#local and global methods
try:
    from data_utils.preprocess import channels_DEAP
    print("channels_DEAP successfully loaded from LibEER")
except ImportError as e:
    print(f"Error loading LibEER: {e}. Please verify that '/content/LibEER' exists.")

import torch
import os
import importlib
import xai_utils.data_utils
importlib.reload(xai_utils.data_utils)

import method3.model
importlib.reload(method3.model)
from method3.model import integrated_gradients

# 1. Hardware and Model Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model.to(device)
loaded_model.eval()

# 2. Base Parameters
verbose = True
channels = channels_DEAP
subject_id = 0
model_type = loaded_model.__class__.__name__
steps = 50 # Internal steps for Integrated Gradients path integral
save_path = f"/content/drive/MyDrive/Colab Notebooks/plots_{model_type}_results"
os.makedirs(save_path, exist_ok=True)

print(f"Model type detected: {model_type}")
print(f"Results will be saved in: {save_path}")

# ==================== LOCAL MODE (Single Sample) ====================
sample_idx = 0
input_data = X_test_sub0[sample_idx]

args_local = {
    'channels': channels,
    'subject_id': subject_id,
    'sample_idx': sample_idx,
    'input_data': input_data,
    'model_type': model_type,
    'steps': steps,
    'is_global': False # Set to False for single sample attribution
}

print(f"\n1. Running LOCAL Integrated Gradients (Sample {sample_idx})...")
try:
    integrated_gradients(loaded_model, save_path, verbose, args_local)
except Exception as e:
    print(f"Error in Local IG execution: {e}")

# ==================== GLOBAL MODE (Test Set Average) ====================
args_global = {
    'channels': channels,
    'subject_id': subject_id,
    'X_test': X_test_sub0,  # Full test set for averaging
    'y_test': y_test_sub0,  # Ground truth labels to verify correct predictions
    'model_type': model_type,
    'steps': steps,
    'is_global': True,      # Set to True for dataset-wide attribution
    'target_class': 2       # Target class to explain (usually 0 or 1 for DEAP)
}

print("\n2. Running GLOBAL Integrated Gradients (Iterating through Test Set)...")
try:
    # This method utilizes the tqdm progress bar defined in method3.model
    integrated_gradients(loaded_model, save_path, verbose, args_global)
    print(f"\nGlobal process completed. Results saved in: {save_path}")
except Exception as e:
    print(f"Error in Global IG execution: {e}")
    import traceback
    traceback.print_exc()

channels_DEAP successfully loaded from LibEER
Model type detected: DGCNN
Results will be saved in: /content/drive/MyDrive/Colab Notebooks/plots_DGCNN_results

1. Running LOCAL Integrated Gradients (Sample 0)...
Before plot: 0.0 1.0

--- INTEGRATED GRADIENTS STATISTICS (DGCNN - Local) ---
Class: 2
Range: [0.0000, 1.0000]
Global Mean: 0.2435

--- TOP 5 CHANNELS ---
  af3: 0.4768
  cp5: 0.4428
  fc1: 0.4330
  f4: 0.4287
  p7: 0.4262

--- BAND IMPORTANCE ---
  Delta: 0.2774
  Theta: 0.1723
  Alpha: 0.2352
  Beta: 0.1986
  Gamma: 0.3340

2. Running GLOBAL Integrated Gradients (Iterating through Test Set)...


100%|██████████| 586/586 [00:00<00:00, 996.30it/s]



Averaged over 173 samples.
Before plot: 0.0027368828 0.26575518
After plot norm: 0.0 1.0

--- INTEGRATED GRADIENTS STATISTICS (DGCNN - Global) ---
Class: 2
Range: [0.0027, 0.2658]
Global Mean: 0.0609

--- TOP 5 CHANNELS ---
  p7: 0.1266
  af3: 0.1183
  fc1: 0.1136
  fc6: 0.1080
  c3: 0.0916

--- BAND IMPORTANCE ---
  Delta: 0.0654
  Theta: 0.0426
  Alpha: 0.0534
  Beta: 0.0530
  Gamma: 0.0900

Global process completed. Results saved in: /content/drive/MyDrive/Colab Notebooks/plots_DGCNN_results


###Smooth Grad

In [35]:
##SMOOTH GRAD, both local and global methods

try:
    from data_utils.preprocess import channels_DEAP
    print("channels_DEAP successfully loaded from LibEER")
except ImportError as e:
    print(f"Error loading LibEER: {e}. Ensure '/content/LibEER' exists.")

import torch
import os
import importlib

# Correctly reloading the SmoothGrad module
import method4.model
importlib.reload(method4.model)
from method4.model import smooth_grad

# 1. Hardware and Model Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model.to(device)
loaded_model.eval()

# 2. Base Parameters
verbose = True
channels = channels_DEAP
subject_id = 0
model_type = loaded_model.__class__.__name__

# SmoothGrad specific parameters
n_noise_samples = 50   # Number of noisy samples per input
stdev_spread = 0.15    # Noise level (15% of the signal range)

save_path = f"/content/drive/MyDrive/Colab Notebooks/plots_{model_type}_results"
os.makedirs(save_path, exist_ok=True)

print(f"Model type detected: {model_type}")
print(f"Results folder: {save_path}")

# ==================== LOCAL MODE (Single Sample) ====================
sample_idx = 0
input_data = X_test_sub0[sample_idx]

args_local = {
    'channels': channels,
    'subject_id': subject_id,
    'sample_idx': sample_idx,
    'input_data': input_data,
    'model_type': model_type,
    'n_samples': n_noise_samples,
    'stdev_spread': stdev_spread,
    'is_global': False
}

print(f"\n1. Running LOCAL SmoothGrad (Sample {sample_idx})...")
try:
    smooth_grad(loaded_model, save_path, verbose, args_local)
except Exception as e:
    print(f"Error in Local SmoothGrad: {e}")
    import traceback
    traceback.print_exc()

# ==================== GLOBAL MODE (Average over Test Set) ====================
args_global = {
    'channels': channels,
    'subject_id': subject_id,
    'X_test': X_test_sub0,
    'y_test': y_test_sub0,
    'model_type': model_type,
    'n_samples': n_noise_samples,
    'stdev_spread': stdev_spread,
    'is_global': True,
    'target_class': 2       # Ensure this matches your actual class labels (usually 0 or 1)
}

print("\n2. Running GLOBAL SmoothGrad (Iterating through Test Set)...")
try:
    # This will show the tqdm progress bar defined in your model.py
    smooth_grad(loaded_model, save_path, verbose, args_global)
    print(f"\nExecution finished. Results saved in: {save_path}")
except Exception as e:
    print(f"Error in Global SmoothGrad: {e}")
    import traceback
    traceback.print_exc()

channels_DEAP successfully loaded from LibEER
Model type detected: DGCNN
Results folder: /content/drive/MyDrive/Colab Notebooks/plots_DGCNN_results

1. Running LOCAL SmoothGrad (Sample 0)...
Before plot: 0.0 1.0

--- SmoothGrad STATISTICS (DGCNN - Local) ---
Target/Pred Class: 2
Noise Samples per Input: 50
Global Mean Attribution: 0.0912

--- TOP 5 CHANNELS ---
  fc1: 0.2926
  p7: 0.2709
  fc6: 0.2119
  af3: 0.1989
  c3: 0.1902

--- BAND IMPORTANCE ---
  Delta: 0.0814
  Theta: 0.0500
  Alpha: 0.0434
  Beta: 0.0734
  Gamma: 0.2079

2. Running GLOBAL SmoothGrad (Iterating through Test Set)...
Starting Global SmoothGrad for Subject 0...


100%|██████████| 586/586 [00:21<00:00, 27.64it/s]



Averaged over 173 samples.
Before plot: 0.04133383 17.89509
After plot norm: 0.0 1.0

--- SmoothGrad STATISTICS (DGCNN - Global) ---
Target/Pred Class: 2
Noise Samples per Input: 50
Global Mean Attribution: 1.5450

--- TOP 5 CHANNELS ---
  fc1: 5.6640
  p7: 4.7082
  fc6: 3.4852
  c3: 3.3705
  af3: 3.0933

--- BAND IMPORTANCE ---
  Delta: 1.4268
  Theta: 0.8722
  Alpha: 0.6840
  Beta: 1.2586
  Gamma: 3.4834

Execution finished. Results saved in: /content/drive/MyDrive/Colab Notebooks/plots_DGCNN_results
